In [1]:
import pandas as pd
import numpy as np

In [2]:
#Load merged rE2G predictions (see Nakato_prep_data_1.ipynb)
E2G = pd.read_table("merge_rE2G_retina.bed", header=None, sep="\t")
column_names = pd.read_table("ENCFF182EGZ.bed", header=None, sep="\t", nrows=1).iloc[0].tolist()
column_names[0] = "chrom"
E2G.columns = column_names
E2G.head()

,chrom,start,end,name,class,TargetGene,TargetGeneEnsemblID,TargetGeneTSS,isSelfPromoter,CellType,distanceToTSS.Feature,normalizedDNase_prom.Feature,3DContact.Feature,ABC.Score.Feature,numCandidateEnhGene.Feature,numTSSEnhGene.Feature,sumNearbyEnhancers.Feature,ubiquitousExpressedGene.Feature,Score
0,chr1,778471,778971,chr1:778471-778971,intergenic,LINC00115,ENSG00000225880,827522,False,retina,48801.0,0.404094,0.099047,0.292515,1,4,0.0,False,0.852104
1,chr1,778471,778971,chr1:778471-778971,intergenic,LINC01128,ENSG00000228794,827590,False,retina,48869.0,0.403658,0.099059,0.292532,1,4,0.0,False,0.852220
2,chr1,778481,778981,chr1:778481-778981,intergenic,LINC00115,ENSG00000225880,827522,False,retina,48791.0,0.429691,0.099046,0.241150,1,4,0.0,False,0.766227
3,chr1,778481,778981,chr1:778481-778981,intergenic,LINC01128,ENSG00000228794,827590,False,retina,48859.0,0.429691,0.099057,0.241166,1,4,0.0,False,0.766054
4,chr1,778483,778983,chr1:778483-778983,intergenic,LINC01128,ENSG00000228794,827590,False,retina,48857.0,0.399545,0.099057,0.282739,1,4,0.0,False,0.844200


In [3]:
#Load output from DE_Nakato_2.R
Nakato_deseq = pd.read_csv("GSE196450_RNAseq_all_DE.txt", sep="\t")
Nakato_deseq = Nakato_deseq.rename(columns={'Gene': 'TargetGene'})
E2G_DE_merge = pd.merge(Nakato_deseq, E2G, on="TargetGene", how="inner", suffixes=["_gene", "_enhancer"])  
E2G_DE_merge.head()

,TargetGene,ensembl_id,chromosome,strand,start_gene,end_gene,length,type,RAD21,NIPBL,...,CellType,distanceToTSS.Feature,normalizedDNase_prom.Feature,3DContact.Feature,ABC.Score.Feature,numCandidateEnhGene.Feature,numTSSEnhGene.Feature,sumNearbyEnhancers.Feature,ubiquitousExpressedGene.Feature,Score
0,A3GALT2,ENSG00000184389,chr1,-,33306765,33321098,14333,protein_coding,NaN,NaN,...,retinal pigment epithelial cell,221615.0,0.326454,0.006700,0.084885,11,3,0.000000,False,0.856397
1,A3GALT2,ENSG00000184389,chr1,-,33306765,33321098,14333,protein_coding,NaN,NaN,...,retinal pigment epithelial cell,193719.0,0.326454,0.008012,0.025032,10,3,0.000000,False,0.334526
2,A3GALT2,ENSG00000184389,chr1,-,33306765,33321098,14333,protein_coding,NaN,NaN,...,retinal pigment epithelial cell,99875.5,0.326454,0.012987,0.012395,6,2,159.836589,False,0.288673
3,A3GALT2,ENSG00000184389,chr1,-,33306765,33321098,14333,protein_coding,NaN,NaN,...,retinal pigment epithelial cell,20224.0,0.326454,0.047294,0.015794,1,1,65.952288,False,0.334779
4,A3GALT2,ENSG00000184389,chr1,-,33306765,33321098,14333,protein_coding,NaN,NaN,...,retinal pigment epithelial cell,0.0,0.326454,0.224400,1.000000,0,0,0.000000,False,1.000000


In [4]:
#This strategy is equivalent to that in Tjalsma et al. 2025 (https://github.com/deLaatLab/Tjalsma_2025/blob/main/Bru-seq/e2g/rna_e2g_cleaned.R) 

###Long-range
E2G_DE_merge_long = E2G_DE_merge[
    (E2G_DE_merge['distanceToTSS.Feature'] >= 50_000) &
    (E2G_DE_merge['distanceToTSS.Feature'] < 2_000_000) &
    (E2G_DE_merge['Score'] >= 0.8)
]

#Short-range
E2G_DE_merge_short = E2G_DE_merge[
    (E2G_DE_merge['distanceToTSS.Feature'] > 2000) &
    (E2G_DE_merge['distanceToTSS.Feature'] < 10_000) &
    (E2G_DE_merge['Score'] >= 0.8)
]

#Filter short-range genes not to have a long-range enhancer
E2G_DE_merge_stringent_nomid = E2G_DE_merge[
    (E2G_DE_merge['distanceToTSS.Feature'] >= 10_000) &
    (E2G_DE_merge['Score'] >= 0.3)
]

E2G_DE_merge_short = E2G_DE_merge_short[
    ~E2G_DE_merge_short['TargetGene'].isin(E2G_DE_merge_stringent_nomid['TargetGene'])
]

#Mid-range
E2G_DE_merge_mid = E2G_DE_merge[
    (E2G_DE_merge['distanceToTSS.Feature'] >= 10_000) &
    (E2G_DE_merge['distanceToTSS.Feature'] < 50_000) &
    (E2G_DE_merge['Score'] >= 0.8)
]


#Filter mid-range genes not to have a long-range enhancer
E2G_DE_merge_stringent_nofar = E2G_DE_merge[
    (E2G_DE_merge['distanceToTSS.Feature'] >= 50_000) &
    (E2G_DE_merge['Score'] >= 0.3)
]

E2G_DE_merge_mid = E2G_DE_merge_mid[
    ~E2G_DE_merge_mid['TargetGene'].isin(E2G_DE_merge_long['TargetGene']) &
    ~E2G_DE_merge_mid['TargetGene'].isin(E2G_DE_merge_stringent_nofar['TargetGene'])
]

#No enhancer
E2G_DE_merge_all_enhancers = E2G_DE_merge[
    (E2G_DE_merge['distanceToTSS.Feature'] > 2000) &
    (E2G_DE_merge['Score'] >= 0.3)
]

E2G_DE_merge_none_enhancers = E2G_DE_merge[
    ~E2G_DE_merge['TargetGene'].isin(E2G_DE_merge_all_enhancers['TargetGene'])
]

In [5]:
#Gene categories
long_genes = E2G_DE_merge[
    E2G_DE_merge['TargetGene'].isin(E2G_DE_merge_long['TargetGene'])
]
mid_genes = E2G_DE_merge[
    E2G_DE_merge['TargetGene'].isin(E2G_DE_merge_mid['TargetGene'])
]
short_genes = E2G_DE_merge[
    E2G_DE_merge['TargetGene'].isin(E2G_DE_merge_short['TargetGene'])
]
no_enhancers = E2G_DE_merge[
    E2G_DE_merge['TargetGene'].isin(E2G_DE_merge_none_enhancers['TargetGene'])
]

#For each gene, pick the row with the max Score
def get_top_enhancers(df, category):
    top = df.loc[df.groupby('TargetGene')['Score'].idxmax()].copy()
    top['Category'] = category
    return top

long_top = get_top_enhancers(E2G_DE_merge_long, "Long")
mid_top = get_top_enhancers(E2G_DE_merge_mid, "Mid")
short_top = get_top_enhancers(E2G_DE_merge_short, "Short")

#For "no enhancer" genes, take the first row per TargetGene
no_enh_top = (
    E2G_DE_merge_none_enhancers
    .groupby('TargetGene')
    .head(1)
    .copy()
)
no_enh_top['Category'] = 'NoEnh'

#Combine all
data_selection_500_topenhancers = pd.concat(
    [long_top, mid_top, short_top, no_enh_top],
    ignore_index=True
).drop_duplicates()

#Keep only dysregulated genes
data_selection_500_topenhancers = data_selection_500_topenhancers[data_selection_500_topenhancers["Cluster"]!="None"]

In [6]:
#Show numbers in each group
data_selection_500_topenhancers.groupby("Category", as_index=False).size()

,Category,size
0,Long,502
1,Mid,203
2,NoEnh,822
3,Short,145


In [7]:
data_selection_500_topenhancers.shape

(1672, 95)

In [8]:
data_selection_500_topenhancers.to_csv("GSE196450_DE_E2G.txt", sep="\t", index=False)